# 00 Data + baselines
Download AFHQ, inspect fixed test masks, run the sanity baselines (grey fill, mirror paste).

| Notebook | Pipeline | Code |
|---|---|---|
| `01_mae` | MAE | `src/methods/mae.py` |
| `02_stylegan_inversion` | StyleGAN inversion | `src/methods/stylegan_inversion.py` |
| `03_stylegan_noninversion` | StyleGAN non-inversion | `src/methods/stylegan_noninversion.py` |
| `04_mirror_conditioned` | Mirror-conditioned StyleGAN | `src/methods/mirror_conditioned.py` |
| `05_repaint` | RePaint | `src/methods/repaint.py` |
| `06_mirror_guided_repaint` | Mirror-guided RePaint (ours) | `src/methods/mirror_diffusion.py` |
| `07_results` | comparison tables | `src/eval/results.py` |

Shared code: `src/data/` (preprocessing, masks) and `src/eval/` (metrics, evaluation loop).

## 0. Setup

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/ericacleo/dlpastia.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm
print(os.getcwd())

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: print('NO GPU: Colab -> Runtime > Change runtime type > T4 GPU, then rerun')

## 1. Download data
Set `KAGGLE_API_TOKEN` (Colab: Secrets panel; local: env var). Never commit it. On a Kaggle notebook, add the dataset as an input and symlink it instead.

In [ ]:
# os.environ['KAGGLE_API_TOKEN'] = '...'   # or use Colab secrets
!{sys.executable} scripts/download_data.py
!{sys.executable} scripts/prepare_data.py

## 2. Data + fixed masks (`src/data/`)

In [ ]:
import torch, matplotlib.pyplot as plt
from src.data import get_datasets, load_config
cfg = load_config()
train_ds, test_ds = get_datasets(cfg)
print(len(train_ds), len(test_ds))
fig, ax = plt.subplots(2, 6, figsize=(14,5))
for k in range(6):
    x, m, xm = test_ds[k*200]
    ax[0,k].imshow(((x+1)/2).permute(1,2,0)); ax[1,k].imshow(((xm+1)/2).permute(1,2,0))
    ax[0,k].axis('off'); ax[1,k].axis('off')
plt.show()

## 3. Evaluate (`src/eval/`)
Baselines first: they check the pipeline and give lower bounds. `MirrorPaste` shows what raw left-right symmetry is worth with no learning.

In [ ]:
from src.eval import evaluate
from src.methods import GreyFill, MirrorPaste
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
for mask in ['square', 'one_sided', 'centre']:
    for M in (GreyFill, MirrorPaste):
        print(mask, M.name, evaluate(M(), mask, n_images=100, device=dev))